# Basic Modules


In [2]:
import numpy as np

## DropOut
Randomly set the elements to zero based on given probability. 

In [3]:
class DropOut:
    def __init__(self, p:float=0.1):
        self.p = p
        self.scale = 1. / (1. - self.p)
        self.mask = None

    def forward(self, x:np.ndarray, train:bool=True) -> np.ndarray:
        
        if train:
            self.mask = np.random.rand(*x.shape) > self.p
            return x * self.mask * self.scale
        else:
            return x
    
    def backward(self, dout:np.ndarray) -> np.ndarray:
        return dout * self.mask * self.scale
    


In [4]:
# add some testing code here

dropout_layer = DropOut(p=0.5)
x = np.array([[1.0, 2.0], [3.0, 4.0]])
train_output = dropout_layer.forward(x, train=True)
print("Train output:")
print(train_output)
test_output = dropout_layer.forward(x, train=False)
print("Test output:")
print(test_output)
grad_output = np.array([[0.1, 0.2], [0.3, 0.4]])
grad_input = dropout_layer.backward(grad_output)
print("Gradient input:")
print(grad_input)

Train output:
[[2. 0.]
 [6. 8.]]
Test output:
[[1. 2.]
 [3. 4.]]
Gradient input:
[[0.2 0. ]
 [0.6 0.8]]


## Batch Norm


In [5]:
class BatchNorm:
    def __init__(self, num_features: int, eps: float = 1e-5, momentum: float = 0.1):
        self.num_features = num_features
        self.eps = eps
        self.momentum = momentum

        # learnable parameters
        self.gamma = np.ones(num_features)   # scale
        self.beta = np.zeros(num_features)   # shift

        # running statistics for inference
        self.running_mean = np.zeros(num_features)
        self.running_var = np.ones(num_features)

        # gradients for learnable parameters
        self.dgamma = None
        self.dbeta = None

    def forward(self, x: np.ndarray, train: bool = True) -> np.ndarray:
        """
        x: (N, D) where D == num_features
        """
        if train:
            mean = x.mean(axis=0)                          # (D,)
            var = x.var(axis=0)                            # (D,)

            # update running statistics
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * mean
            self.running_var = (1 - self.momentum) * self.running_var + self.momentum * var

            # normalize
            self.x_hat = (x - mean) / np.sqrt(var + self.eps)   # (N, D)
        else:
            self.x_hat = (x - self.running_mean) / np.sqrt(self.running_var + self.eps)

        # cache for backward
        self.x = x
        self.mean = mean if train else self.running_mean
        self.var = var if train else self.running_var
        self.N = x.shape[0]

        out = self.gamma * self.x_hat + self.beta          # (N, D)
        return out

    def backward(self, dout: np.ndarray) -> np.ndarray:
        """
        dout: (N, D) gradient from upstream
        returns: dx (N, D)
        """
        N = self.N

        # gradients of learnable parameters
        self.dgamma = np.sum(dout * self.x_hat, axis=0)   # (D,)
        self.dbeta = np.sum(dout, axis=0)                  # (D,)

        # gradient through normalization
        dx_hat = dout * self.gamma                         # (N, D)
        std_inv = 1.0 / np.sqrt(self.var + self.eps)      # (D,)

        dx = (1.0 / N) * std_inv * (
            N * dx_hat
            - np.sum(dx_hat, axis=0)
            - self.x_hat * np.sum(dx_hat * self.x_hat, axis=0)
        )
        return dx


## LayerNorm

Layer norm is 

In [7]:
class LayerNorm:
    def __init__(self, normalized_shape, eps: float = 1e-5):
        """
        normalized_shape: int or tuple of ints for the last dimension(s) to normalize over.
                          e.g. (D,) for input (N, D) or (H, W) for input (N, H, W)
        """
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)
        self.normalized_shape = tuple(normalized_shape)
        self.eps = eps

        # learnable parameters, same shape as normalized dimensions
        self.gamma = np.ones(self.normalized_shape)
        self.beta = np.zeros(self.normalized_shape)

        # gradients
        self.dgamma = None
        self.dbeta = None

    def forward(self, x: np.ndarray) -> np.ndarray:
        """
        x: arbitrary shape (..., *normalized_shape)
        Normalizes over the last len(normalized_shape) dimensions.
        """
        self.x = x
        # axes to reduce over: the last len(normalized_shape) dims
        ndim = len(self.normalized_shape)
        self.norm_axes = tuple(range(-ndim, 0))

        self.mean = x.mean(axis=self.norm_axes, keepdims=True)
        self.var = x.var(axis=self.norm_axes, keepdims=True)
        self.x_hat = (x - self.mean) / np.sqrt(self.var + self.eps)

        out = self.gamma * self.x_hat + self.beta
        return out

    def backward(self, dout: np.ndarray) -> np.ndarray:
        """
        dout: same shape as x
        returns: dx, same shape as x
        """
        # number of elements in the normalized dimensions
        n = 1
        for s in self.normalized_shape:
            n *= s

        # gradients of learnable parameters: sum over batch (all non-normalized dims)
        batch_axes = tuple(range(len(dout.shape) - len(self.normalized_shape)))
        self.dgamma = np.sum(dout * self.x_hat, axis=batch_axes)
        self.dbeta = np.sum(dout, axis=batch_axes)

        # gradient through normalization
        dx_hat = dout * self.gamma
        std_inv = 1.0 / np.sqrt(self.var + self.eps)

        dx = (1.0 / n) * std_inv * (
            n * dx_hat
            - dx_hat.sum(axis=self.norm_axes, keepdims=True)
            - self.x_hat * (dx_hat * self.x_hat).sum(axis=self.norm_axes, keepdims=True)
        )
        return dx


## RMSNorm

RMSNorm rescales each input using its root mean square (RMS), without subtracting the mean:

$$\operatorname{RMS}(x) = \sqrt{\frac{1}{D}\sum_{i=1}^{D}x_i^2 + \epsilon}$$

$$y_i = \gamma_i \frac{x_i}{\operatorname{RMS}(x)}$$

Unlike LayerNorm, RMSNorm does **not** center the input and usually has no learned bias $\beta$. It preserves the direction of the input vector while controlling its scale. Removing mean subtraction makes it slightly simpler and cheaper, which is why it is common in modern transformer models.

For an input shaped `(batch, sequence, hidden_size)`, normalization is typically performed over `hidden_size` independently for every token. RMSNorm uses the same computation during training and inference, so it does not need running statistics.

In [9]:
class RMSNorm:
    def __init__(self, normalized_shape, eps: float = 1e-6):
        """Normalize over the last dimension(s) in normalized_shape."""
        if isinstance(normalized_shape, int):
            normalized_shape = (normalized_shape,)

        self.normalized_shape = tuple(normalized_shape)
        self.eps = eps
        self.gamma = np.ones(self.normalized_shape)
        self.dgamma = None

    def forward(self, x: np.ndarray) -> np.ndarray:
        ndim = len(self.normalized_shape)
        self.norm_axes = tuple(range(-ndim, 0))
        self.x = x

        mean_square = np.mean(x ** 2, axis=self.norm_axes, keepdims=True)
        self.inv_rms = 1.0 / np.sqrt(mean_square + self.eps)
        self.x_hat = x * self.inv_rms
        return self.gamma * self.x_hat

    def backward(self, dout: np.ndarray) -> np.ndarray:
        # gamma is shared across all leading batch/sequence dimensions.
        batch_axes = tuple(range(dout.ndim - len(self.normalized_shape)))
        self.dgamma = np.sum(dout * self.x_hat, axis=batch_axes)

        dx_hat = dout * self.gamma
        mean_dx_hat_x = np.mean(
            dx_hat * self.x, axis=self.norm_axes, keepdims=True
        )
        return (
            dx_hat * self.inv_rms
            - self.x * mean_dx_hat_x * self.inv_rms ** 3
        )

In [10]:
# Example: normalize the hidden dimension of each token independently.
rng = np.random.default_rng(0)
x = rng.normal(size=(2, 3, 4))
rms_norm = RMSNorm(normalized_shape=4)

out = rms_norm.forward(x)
dout = rng.normal(size=x.shape)
dx = rms_norm.backward(dout)

print("Input shape:", x.shape)
print("Output RMS per token:\n", np.sqrt(np.mean(out ** 2, axis=-1)))
print("Output mean per token (not forced to zero):\n", out.mean(axis=-1))
print("dx shape:", dx.shape)
print("dgamma shape:", rms_norm.dgamma.shape)

# Numerical check for one input gradient.
index = (0, 0, 0)
h = 1e-5
x_pos, x_neg = x.copy(), x.copy()
x_pos[index] += h
x_neg[index] -= h
loss_pos = np.sum(rms_norm.forward(x_pos) * dout)
loss_neg = np.sum(rms_norm.forward(x_neg) * dout)
numerical_grad = (loss_pos - loss_neg) / (2 * h)

print("Analytical gradient:", dx[index])
print("Numerical gradient: ", numerical_grad)

Input shape: (2, 3, 4)
Output RMS per token:
 [[0.9999956  0.99999934 0.9999992 ]
 [0.99999973 0.99999879 0.99999918]]
Output mean per token (not forced to zero):
 [[ 0.54810088  0.59807938 -0.80888192]
 [-0.82329032  0.23087482  0.29525158]]
dx shape: (2, 3, 4)
dgamma shape: (4,)
Analytical gradient: 3.067729101386586
Numerical gradient:  3.067729101013938
